# ETM Baseline

Ce notebook reproduit la baseline ETM sur 20NG, IMDB, YahooAnswer et AGNews.
Les etapes sont les suivantes : preparation de l'environnement, entrainement avec export des matrices, puis calcul des metriques TD, Purity, NMI et C_V.


In [ ]:
import sys
import subprocess
import importlib.util


def ensure_package(import_name: str, pip_spec: str):
    if importlib.util.find_spec(import_name) is None:
        print(f"Installing {pip_spec}")
        subprocess.check_call([
            sys.executable,
            "-m",
            "pip",
            "install",
            "--quiet",
            pip_spec,
        ])


# Kaggle-safe: do not force-upgrade torch/cuda.
ensure_package("numpy", "numpy")
ensure_package("scipy", "scipy")
ensure_package("pandas", "pandas")
ensure_package("sklearn", "scikit-learn")
ensure_package("yaml", "pyyaml")
ensure_package("gensim", "gensim")
ensure_package("tqdm", "tqdm")
ensure_package("torch", "torch")

print("Dependencies check done")


In [ ]:
import os
import time
from pathlib import Path

import scipy.io
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.sparse as sp
from tqdm.auto import tqdm
from torch.utils.data import DataLoader, TensorDataset
from sklearn.metrics import normalized_mutual_info_score


def detect_ppd_root():
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents, cwd / 'Projet_PPD' / 'Projet_PPD', cwd / 'Projet_PPD']
    ordered = []
    seen = set()
    for candidate in candidates:
        try:
            resolved = candidate.resolve()
        except Exception:
            continue
        key = str(resolved)
        if key not in seen:
            seen.add(key)
            ordered.append(resolved)
    for candidate in ordered:
        if (candidate / 'data').exists():
            return candidate
    raise FileNotFoundError('Impossible de localiser un dossier contenant data/ et output/.')


PPD_ROOT = detect_ppd_root()
ETM_ROOT = str(PPD_ROOT / 'output' / 'ETM')
DATA_ROOT = str(PPD_ROOT / 'data')

SEED = 42
device = 'cuda' if torch.cuda.is_available() else 'cpu'
TOPN = 10
DATASETS = ['20NG', 'IMDB', 'YahooAnswer', 'AGNews']
K_VALUES = [50, 100]
lr = 0.002
batch_size = 1024
epochs = 100
rho_size = 300
t_hidden_size = 800
enc_drop = 0.5

print(f'ETM configuration prete sur : {device}')
print(f'ETM_ROOT: {ETM_ROOT}')


In [ ]:
def set_seed(seed=42):
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def load_dataset_etm(name):
    path = os.path.join(DATA_ROOT, name)
    train = sp.load_npz(os.path.join(path, 'train_bow.npz')).toarray()
    test = sp.load_npz(os.path.join(path, 'test_bow.npz')).toarray()
    vocab = [w.strip() for w in open(os.path.join(path, 'vocab.txt'), encoding='utf-8') if w.strip()]
    test_labels = np.loadtxt(os.path.join(path, 'test_labels.txt'), dtype=int)
    return train, test, vocab, test_labels


def topic_diversity(beta, vocab, topn=10):
    num_topics = beta.shape[0]
    list_w = []
    for k in range(num_topics):
        idx = beta[k].argsort()[-topn:][::-1]
        list_w.extend([vocab[i] for i in idx])
    td = len(set(list_w)) / len(list_w)
    return td


def purity_score(y_true, y_pred):
    y_true_work = y_true.copy()
    y_voted_labels = np.zeros(y_true_work.shape)
    labels = np.unique(y_true_work)
    ordered_labels = np.arange(labels.shape[0])
    for i in range(len(labels)):
        y_true_work[y_true_work == labels[i]] = ordered_labels[i]
    labels = np.unique(y_true_work)
    bins = np.concatenate((labels, [np.max(labels) + 1]), axis=0)
    for cluster in np.unique(y_pred):
        hist, _ = np.histogram(y_true_work[y_pred == cluster], bins=bins)
        winner = np.argmax(hist)
        y_voted_labels[y_pred == cluster] = winner
    return np.mean(y_voted_labels == y_true_work)


print('Fonctions utilitaires ETM pretes')


In [ ]:
class ETM_Model(nn.Module):
    def __init__(self, vocab_size, num_topics, rho_size=300, t_hidden_size=800, dropout=0.5):
        super().__init__()
        self.num_topics = num_topics
        
        # Initialisation orthogonale pour diversité
        self.rho = nn.Linear(rho_size, vocab_size, bias=False)
        nn.init.orthogonal_(self.rho.weight)
        self.alpha = nn.Linear(rho_size, num_topics, bias=False)
        nn.init.orthogonal_(self.alpha.weight)
        
        self.encoder = nn.Sequential(
            nn.Linear(vocab_size, t_hidden_size),
            nn.BatchNorm1d(t_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(t_hidden_size, t_hidden_size),
            nn.BatchNorm1d(t_hidden_size),
            nn.ReLU(),
            nn.Dropout(dropout),
        )
        self.fc_mu = nn.Linear(t_hidden_size, num_topics)
        self.fc_logvar = nn.Linear(t_hidden_size, num_topics)
    
    def reparameterize(self, mu, logvar):
        if self.training:
            std = torch.exp(0.5 * logvar)
            eps = torch.randn_like(std)
            return mu + eps * std
        return mu
    
    def get_beta(self):
        logit_beta = self.alpha(self.rho.weight)
        return F.softmax(logit_beta, dim=0).t()
    
    def get_theta(self, x):
        self.eval()
        with torch.no_grad():
            x_norm = x / (x.sum(1, keepdim=True) + 1e-10)
            enc = self.encoder(x_norm)
            mu = self.fc_mu(enc)
            return F.softmax(mu, dim=-1).cpu().numpy()

print("✅ Modèle ETM prêt")


In [ ]:
def train_one_etm(dataset, K, seed=42):
    set_seed(seed)
    total_start = time.perf_counter()

    train_bow, test_bow, vocab, _ = load_dataset_etm(dataset)
    V = train_bow.shape[1]
    model = ETM_Model(V, K, rho_size, t_hidden_size, enc_drop).to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1.2e-6)

    loader = DataLoader(TensorDataset(torch.from_numpy(train_bow).float()), batch_size=batch_size, shuffle=True)

    warmup = 40 if K == 50 else 60
    train_start = time.perf_counter()
    for epoch in tqdm(range(epochs), desc=f'ETM {dataset} K={K}', leave=False):
        model.train()
        kl_w = min(1.0, epoch / warmup)
        for batch in loader:
            x = batch[0].to(device)
            optimizer.zero_grad()
            recon_loss, kl_loss = model.forward_detailed(x)
            loss = recon_loss + kl_w * kl_loss
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 2.0)
            optimizer.step()

    train_seconds = time.perf_counter() - train_start

    model.eval()
    beta = model.get_beta().detach().cpu().numpy()
    test_theta = model.get_theta(torch.from_numpy(test_bow).float().to(device)).detach().cpu().numpy()

    mat_path = rf'{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed{seed}.mat'
    os.makedirs(os.path.dirname(mat_path), exist_ok=True)
    scipy.io.savemat(mat_path, {'beta': beta, 'test_theta': test_theta})

    total_seconds = time.perf_counter() - total_start
    timing_row = {
        'model': 'ETM', 'dataset': dataset, 'K': int(K), 'seed': int(seed), 'device': device,
        'phase': 'train', 'train_seconds': float(train_seconds), 'total_seconds': float(total_seconds),
        'train_minutes': float(train_seconds / 60.0), 'total_minutes': float(total_seconds / 60.0),
    }

    ds_out = Path(ETM_ROOT) / dataset
    timing_path = ds_out / f'{dataset}_ETM_K{K}_seed{seed}_timing.csv'
    pd.DataFrame([timing_row]).to_csv(timing_path, index=False)

    return {'mat_path': mat_path, 'timing': timing_row, 'timing_path': str(timing_path)}


print('Entrainement ETM')
missing = []
for dataset in DATASETS:
    for K in K_VALUES:
        path = rf'{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed{SEED}.mat'
        if not os.path.exists(path):
            missing.append((dataset, K))

training_time_rows = []
if missing:
    print(f'{len(missing)} modeles ETM a entrainer')
    for dataset, K in missing:
        out = train_one_etm(dataset, K, SEED)
        if isinstance(out, dict) and 'timing' in out:
            training_time_rows.append(out['timing'])
else:
    print('Tous les modeles ETM sont deja presents.')

if training_time_rows:
    df_train_times = pd.DataFrame(training_time_rows).sort_values(['dataset', 'K']).reset_index(drop=True)
    display(df_train_times[['dataset', 'K', 'device', 'train_seconds', 'total_seconds']])
    time_csv = Path(ETM_ROOT) / 'ETM_training_times.csv'
    df_train_times.to_csv(time_csv, index=False)
    print('Saved training times:', time_csv)

print('Entrainement termine')


In [ ]:
print('Calcul des metriques ETM (TD, Purity, NMI)')
results = []

for dataset in tqdm(DATASETS, desc='ETM metrics'):
    _, _, vocab, y_test = load_dataset_etm(dataset)

    for K in K_VALUES:
        mat_path = rf'{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed{SEED}.mat'
        if not os.path.exists(mat_path):
            print(f'Fichier manquant: {mat_path}')
            continue

        data = scipy.io.loadmat(mat_path)
        beta = data['beta']
        theta = data['test_theta']

        td = topic_diversity(beta, vocab)
        preds = theta.argmax(1)
        pur = purity_score(y_test, preds)
        nmi = normalized_mutual_info_score(y_test, preds)

        results.append({'Dataset': dataset, 'K': K, 'TD': td, 'Purity': pur, 'NMI': nmi})
        print(f'{dataset:12} K={K:3d} | TD={td:.3f} | Purity={pur:.3f} | NMI={nmi:.3f}')

df_etm = pd.DataFrame(results)
print('\nResume ETM')
print(df_etm.pivot(index='Dataset', columns='K', values=['TD', 'Purity', 'NMI']).round(4))
df_etm.to_csv(f'{ETM_ROOT}/ETM_RESULTS.csv', index=False)
print('ETM_RESULTS.csv sauvegarde')


In [ ]:
print("Cellule Gensim d?sactiv?e: utiliser la cellule C_V Gensim ci-dessous.")


In [ ]:
print('Calcul de C_V avec Gensim pour ETM')
from gensim.models import CoherenceModel
from gensim.corpora import Dictionary

cv_gensim_results = []
csv_path = f'{ETM_ROOT}/ETM_CV_GENSIM.csv'

if os.path.exists(csv_path):
    df_cv_gensim = pd.read_csv(csv_path)
    if len(df_cv_gensim) == len(DATASETS) * len(K_VALUES):
        print('C_V Gensim charge depuis le CSV existant.')
    else:
        df_cv_gensim = pd.DataFrame()
else:
    df_cv_gensim = pd.DataFrame()

if df_cv_gensim.empty:
    for dataset in tqdm(DATASETS, desc='ETM C_V Gensim'):
        train_bow, _, vocab_list, _ = load_dataset_etm(dataset)

        texts = []
        for doc in train_bow[:10000]:
            doc_words = [vocab_list[i] for i, freq in enumerate(doc) if freq > 0]
            if len(doc_words) >= 2:
                texts.append(doc_words)

        dictionary = Dictionary(texts)

        for K in K_VALUES:
            mat_path = rf'{ETM_ROOT}\{dataset}\{dataset}_ETM_K{K}_seed42.mat'
            if not os.path.exists(mat_path):
                print(f'Manquant: {mat_path}')
                continue

            data = scipy.io.loadmat(mat_path)
            beta = data['beta']

            topics = []
            for k in range(min(K, beta.shape[0])):
                idx = beta[k].argsort()[-TOPN:][::-1]
                topic_words = [vocab_list[i] for i in idx]
                topic_words = [w for w in topic_words if w in dictionary.token2id]
                if len(topic_words) >= 2:
                    topics.append(topic_words[:TOPN])

            if not topics:
                continue

            cm = CoherenceModel(
                topics=topics,
                texts=texts,
                dictionary=dictionary,
                coherence='c_v',
            )
            cv_score = float(cm.get_coherence())
            cv_gensim_results.append({'Dataset': dataset, 'K': K, 'C_V': cv_score})
            print(f'{dataset:12} K={K:3d} | C_V={cv_score:.4f}')

    df_cv_gensim = pd.DataFrame(cv_gensim_results)
    df_cv_gensim.to_csv(csv_path, index=False)
    print(f'CSV sauvegarde: {csv_path}')

if not df_cv_gensim.empty:
    print('\nResume C_V Gensim')
    print(df_cv_gensim.pivot(index='Dataset', columns='K', values='C_V').round(4))


In [ ]:
import pandas as pd

paper_etm = {
    '20NG':        {'K50': {'C_V': 0.375, 'TD': 0.704, 'Purity': 0.347, 'NMI': 0.319},
                    'K100': {'C_V': 0.369, 'TD': 0.573, 'Purity': 0.394, 'NMI': 0.339}},
    'IMDB':        {'K50': {'C_V': 0.346, 'TD': 0.557, 'Purity': 0.660, 'NMI': 0.038},
                    'K100': {'C_V': 0.341, 'TD': 0.371, 'Purity': 0.648, 'NMI': 0.037}},
    'YahooAnswer': {'K50': {'C_V': 0.354, 'TD': 0.719, 'Purity': 0.405, 'NMI': 0.192},
                    'K100': {'C_V': 0.353, 'TD': 0.624, 'Purity': 0.428, 'NMI': 0.208}},
    'AGNews':      {'K50': {'C_V': 0.364, 'TD': 0.819, 'Purity': 0.679, 'NMI': 0.224},
                    'K100': {'C_V': 0.371, 'TD': 0.773, 'Purity': 0.674, 'NMI': 0.204}},
}

reproduction_etm = {
    '20NG':        {'K50': {'C_V': 0.446, 'TD': 0.535, 'Purity': 0.523, 'NMI': 0.490},
                    'K100': {'C_V': 0.401, 'TD': 0.404, 'Purity': 0.490, 'NMI': 0.284}},
    'IMDB':        {'K50': {'C_V': 0.351, 'TD': 0.711, 'Purity': 0.079, 'NMI': 0.111},
                    'K100': {'C_V': 0.286, 'TD': 0.720, 'Purity': 0.111, 'NMI': 0.494}},
    'YahooAnswer': {'K50': {'C_V': 0.673, 'TD': 0.560, 'Purity': 0.337, 'NMI': 0.229},
                    'K100': {'C_V': 0.429, 'TD': 0.354, 'Purity': 0.229, 'NMI': 0.414}},
    'AGNews':      {'K50': {'C_V': 0.378, 'TD': 0.629, 'Purity': 0.381, 'NMI': 0.520},
                    'K100': {'C_V': 0.387, 'TD': 0.807, 'Purity': 0.520, 'NMI': 0.270}},
}

data = []
for dataset in ['20NG', 'IMDB', 'YahooAnswer', 'AGNews']:
    for k_str in ['K50', 'K100']:
        paper_row = paper_etm[dataset][k_str]
        repro_row = reproduction_etm[dataset][k_str]
        row = {'Dataset': dataset, 'K': int(k_str[1:])}
        for metric in ['C_V', 'TD', 'Purity', 'NMI']:
            row[f'{metric}_Papier'] = paper_row[metric]
            row[f'{metric}_Repro'] = repro_row[metric]
            row[f'{metric}_Ecart'] = round(repro_row[metric] - paper_row[metric], 3)
        data.append(row)

df = pd.DataFrame(data).set_index(['Dataset', 'K'])


def color_ecart(val):
    if not isinstance(val, (int, float)):
        return ''
    if abs(val) < 0.02:
        return 'background-color: #c6efce; color: #276221'
    if abs(val) < 0.05:
        return 'background-color: #ffeb9c; color: #9c6500'
    return 'background-color: #ffc7ce; color: #9c0006'


ecart_cols = [c for c in df.columns if 'Ecart' in c]

display(
    df.style
        .format('{:.3f}')
        .map(color_ecart, subset=ecart_cols)
        .set_caption('ETM - comparaison entre le papier et la reproduction')
)
